### Работа с данными

https://pytorch.org/tutorials/beginner/basics/data_tutorial.html

#### 1. Организуем доступ к данным с `torch.utils.data.Dataset`

Датасет в pytorch - это объект класса, в котором реализовано два обязательных метода: `__getitem__(self, index: int)` (получение одиночного примера по индексу) и `__len__(self)` (получение общего количества примеров). Этих методов достаточно, чтобы разбивать датасет на мини-батчи — эту работу делает класс `torch.utils.DataLoader` с помощью различных семплеров, с ними мы познакомимся позже

In [1]:
import torch

torch.manual_seed(42)

In [2]:
from torch.utils.data import Dataset


class MyDataset(Dataset):
    def __init__(self, n: int) -> None:
        super().__init__()
        self.data = torch.arange(n * 3).view((n, 3))
        self.labels = torch.randint(0, 5, size=(n,))

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.data[index], self.labels[index]

    def __len__(self) -> int:
        return len(self.data)


dataset = MyDataset(n=10)
print(dataset[0])
print(len(dataset))

(tensor([0, 1, 2]), tensor(2))
10


#### 2. Пакуем данные в батчи с `torch.utils.data.Dataloader`

У `torch.utils.data.DataLoader` много аргументов, на практике чаще всего используются
- `dataset` — объект, поддерживающий методы `__getitem__` и `__len__` (вопрос: можно ли передать список? словарь? множество?)
- `batch_size` — размер мини-батча
- `shuffle` — нужно ли перетасовать индексы перед нарезкой на мини-батчи (это всегда стоит делать с обучающими данными, почему?)
- `num_workers` — количество процессов, которые будут загружать данные — иногда позволяет ускорить обучение (подумайте, в каком случае?)

In [3]:
from torch.utils.data import DataLoader

my_loader = DataLoader(
    dataset=dataset,
    batch_size=4,
    shuffle=True,
)

In [4]:
# получим первый батч
x, y = next(iter(my_loader))
print(f"x shape: {x.shape}")  # (batch_size, n_features)
print(f"y shape: {y.shape}")  # (batch_size,)
print(x)
print(y)

x shape: torch.Size([4, 3])
y shape: torch.Size([4])
tensor([[ 0,  1,  2],
        [12, 13, 14],
        [ 9, 10, 11],
        [27, 28, 29]])
tensor([2, 1, 4, 3])


По умолчанию `DataLoader` просто складывает примеры в один тензор через `torch.stack`. Если примеры имеют разную длину (например, последовательности белков или ридов), стандартная склейка не сработает — в этом случае можно передать свою функцию `collate_fn`, которая определяет, как собрать список примеров в батч (например, дополнить короткие последовательности нулями).

In [7]:
# пример: последовательности разной длины
sequences = [
    torch.tensor([1, 2, 3]),
    torch.tensor([4, 5]),
    torch.tensor([6, 7, 8, 9]),
]

def pad_collate(batch):
    """Дополняет последовательности нулями до длины самой длинной в батче."""
    max_len = max(len(seq) for seq in batch)
    padded = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, seq in enumerate(batch):
        padded[i, :len(seq)] = seq
    return padded

loader = DataLoader(sequences, batch_size=3, collate_fn=pad_collate)
print(next(iter(loader)))

tensor([[1, 2, 3, 0],
        [4, 5, 0, 0],
        [6, 7, 8, 9]])
